# E5–E6: knowledge as an object that can be changed and reused
CPU-only, small enough to understand every line. This is a research notebook, not an ARC or SOTA result.

**Distinction:** recurrence in numerical state, executable composition, invention of reusable definitions, revision of definitions, and learning revision operators are different capabilities. Pāṇini's grammar provides inspiration for definitions (saṃjñā), inherited scope (anuvṛtti), general rules and exceptions (utsarga/apavāda), and metarules (paribhāṣā). It is not itself a learning algorithm; interpretations of rule ordering are contested. See [Rajpopat](https://www.repository.cam.ac.uk/items/50535a1c-f264-40bd-9198-08693d3ac325), [Kiparsky](https://journals.akademicka.pl/cis/article/view/7558), [DreamCoder](https://arxiv.org/abs/2006.08381), [Stitch](https://arxiv.org/abs/2211.16605), [Doyle's truth maintenance](https://dspace.mit.edu/entities/publication/5377b306-4ecc-4687-b1f5-78cbb4a0543a), and [ILP survey](https://doi.org/10.1007/s10994-021-06089-1).

E5 isolates ambiguity about the scope of revision. E6 isolates the acquisition of executable definitions as operands of later synthesis. Neither claims novelty over the literature. The eventual hypothesis is their combination: discover definitions; retain dependencies and rival theories; learn how to transform theories; validate the change and revalidate its dependents. Do not call a learned prior a discovered metarule, or call a cached truth table general intelligence.

## E5 · A contradiction admits multiple explanations
Four actions depend on a shared predicate. A new observed effect may require editing the shared definition (all actions) or adding an exception (one action). The learner is not told the scope. Work in residual bits relative to the old theory: 1 = flip the old prediction. Candidate predicates are supplied; this experiment does not invent them.

**Expected behavior:** evidence for a shared edit should change predictions for unobserved actions. A local exception should not. A learned preference for global edits should help only if the task distribution supports it. The oracle scope-*frequency* prior is a ceiling diagnostic, not an implementable baseline. No GPU.

In [ ]:
import numpy as np

A, X = 4, 16
EPS = .02  # declared observation-noise model, not numerical clipping

def predicates(domain):
    x = np.arange(X)
    if domain == 'threshold':
        return np.array([x >= k for k in (3, 6, 9, 12)] + [x % 2 == r for r in (0, 1)])
    return np.array([((x >> b) & 1) == 1 for b in range(4)] + [(x & m) == m for m in (3, 12)])

def theories(domain):
    # One shared definition revision or one action-specific exception.
    p = predicates(domain)
    return np.array([np.broadcast_to(t, (A, X)) if scope == -1 else
                     (np.arange(A)[:, None] == scope) & t[None, :]
                     for scope in [-1, *range(A)] for t in p], dtype=float).reshape(-1, A*X)

def prior(H, pg):
    k = len(H)//(A+1)
    return np.r_[np.full(k, pg/k), np.full(A*k, (1-pg)/(A*k))]

def episode(H, pg, rng, outside=False):
    t = H[rng.choice(len(H), p=prior(H, pg))].copy()
    if outside: # two independent local exceptions, not in the single-edit hypothesis family
        k = len(H)//(A+1)
        ids = rng.choice(A, 2, replace=False)
        t = np.maximum(H[k*(ids[0]+1)+rng.integers(k)], H[k*(ids[1]+1)+rng.integers(k)])
    q = rng.permutation(A*X)
    return t, q

# First observation cannot distinguish global change from local exception.
H=theories('threshold'); k=len(H)//(A+1)
global_edit, local_edit = H[0], H[k]
assert global_edit[3]==local_edit[3]==1
assert global_edit[X+3]==1 and local_edit[X+3]==0
w=prior(H,.5); assert np.isclose(w.sum(),1)
print('one observation agrees; the second action distinguishes the theories')

### Learn which theory edit is plausible
Marginalize the unknown edit type and predicate, rather than assigning a single explanation prematurely. Learn one scalar scope prior from 60 independent training episodes, each with eight observations. Then transfer it to a different predicate vocabulary. This is ordinary hierarchical Bayesian inference, deliberately a minimal control for future learned revision policies.

In [ ]:
def likelihood(H, q, y):
    return np.prod(np.where(H[:, q] == y, 1-EPS, EPS), axis=1)

def fit_scope(H, rng, n=60, pg=.9):
    # Marginal likelihood of observed outcomes; no oracle labels for edit type.
    L = []
    for _ in range(n):
        t,q = episode(H, pg, rng); q=q[:8]
        L.append(likelihood(H, q, t[q]))
    grid = np.linspace(.05,.95,91)
    ll = np.log(np.array(L) @ np.stack([prior(H,p) for p in grid]).T).sum(0)
    return grid[ll.argmax()]

def posterior(H,w,q,y):
    z=w*likelihood(H,q,y)
    return z/z.sum()

def predict(H, w, q, y, independent=False):
    if not independent: return posterior(H,w,q,y) @ H
    # Same marginal prior and same evidence, but cut cross-action dependencies.
    out = np.empty(A*X)
    for a in range(A):
        ix = q//X == a
        out[a*X:(a+1)*X] = posterior(H,w,q[ix],y[ix]) @ H[:,a*X:(a+1)*X]
    return out

assert np.allclose(predict(H,w,np.array([],int),np.array([])),w@H)

### Falsify transfer rather than just demonstrate it
Evaluate unobserved queries after 1,2,4,8 observations. Same random streams for all methods. Independent model gets the learned marginal prior but no cross-action updates. Joint uses fixed 50/50 scope prior. Meta uses learned prior. Oracle knows generating scope frequency, not the actual edit. Stress cases: reverse scope frequencies; two simultaneous exceptions outside candidate family. Brier loss measures accuracy and confidence (lower is better); bootstrap over episodes, not individual cells.

In [ ]:
def evaluate(domain, pg, learned, seed, outside=False, n=300):
    H=theories(domain); rng=np.random.default_rng(seed)
    scores={name:[] for name in ['independent','joint','meta','oracle']}
    for _ in range(n):
        t,q=episode(H,pg,rng,outside)
        for k in (1,2,4,8):
            seen=q[:k]; test=q[k:]; y=t[seen]
            for name,p,ind in [('independent',learned,True),('joint',.5,False),('meta',learned,False),('oracle',pg,False)]:
                pred=predict(H,prior(H,p),seen,y,ind)[test]
                scores[name].append([k,np.mean((pred-t[test])**2),np.mean((pred>=.5)==t[test])])
    return {name:np.array(v).reshape(n,4,3) for name,v in scores.items()}

for seed in (137, 291):
    learned=fit_scope(H,np.random.default_rng(seed))
    print('seed',seed,'learned global probability',round(learned,2))
    for domain,pg,outside in [('threshold',.9,False),('bits',.9,False),('bits',.1,False),('bits',.5,True)]:
        s=evaluate(domain,pg,learned,seed+1000,outside)
        print(domain,pg,'outside',outside, 'Brier after 1,2,4,8:')
        for name,v in s.items(): print(name, np.round(v[:,:,1].mean(0),4))
        d=s['joint'][:,2,1]-s['meta'][:,2,1]
        boot=np.random.default_rng(42).choice(d,(2000,len(d)),replace=True).mean(1)
        print('meta improvement at 4:',round(d.mean(),4),np.round(np.quantile(boot,[.025,.975]),4))

### Captured standalone CPU output — E5
Notebook cells are not marked executed. The same functions were run as a Python script. Improvement = fixed joint Brier minus learned joint Brier (positive is good).

| Condition | Seed 137 | Seed 291 |
|---|---:|---:|
| Same vocabulary, matched scope | +0.0186 | +0.0111 |
| New vocabulary, matched scope | +0.0151 | +0.0126 |
| New vocabulary, reversed scope | -0.0560 | -0.0620 |
| Two edits outside model | -0.0409 | -0.0432 |

Learned global probabilities .92 and .93. Matched cross-vocabulary 95% episode-bootstrap intervals [0.0102,0.0204] and [0.0066,0.0182]. Reversed scope intervals entirely negative. Shared dependencies help transfer; overconfident meta-prior hurts when wrong. This proves neither real-world concept discovery nor a new learning algorithm. Predicates and scopes are provided, tasks noiseless despite a small likelihood noise model, and there is no within-episode environmental change. The next test must consider multiple revisions and explicit old-evidence compatibility.

## E6 · The result of synthesis becomes an operand of synthesis
Unlike E3's fixed bank of 420 transformations, generate Boolean ASTs incrementally. Five primitive inputs, three binary operations and negation; semantic deduplication on the entire finite input domain. Solve a task from its truth table. Name the inferred program and make it available as a primitive for later tasks, including programs that depend on other inferred programs.

This is a **positive-control curriculum**, deliberately constructed to have compositional structure. Full truth-table supervision; no sample-efficiency or generalization-to-unseen-input claim. The flat enumerator sees the same truth tables, with the same budget per task, and can reuse its search algorithm but not named programs. Exact retrieval cannot solve these distinct tables. DreamCoder/Stitch-style learned abstractions and a cached incremental flat enumerator are stronger future controls. Candidate counts are not FLOPs; cached truth tables make execution cheap. All earlier library construction searches are charged in the totals.

**Cell prediction:** the first task costs the same. Once K0 exists, later tasks should become shorter in the expanded language. Irrelevant macros may harm search on an unrelated task. We must test that rather than assume more concepts are always better.

In [ ]:
from time import perf_counter

N=5; FULL=(1<<(1<<N))-1
ATOM={f'x{i}':sum(1<<j for j in range(1<<N) if (j>>i)&1) for i in range(N)}
OPS={'&':lambda a,b:a&b, '|':lambda a,b:a|b, '^':lambda a,b:a^b}

def synth(target, library=(), budget=25000, max_cost=11):
    """Bottom-up semantic enumerator. Cost = AST nodes; named calls cost one.
    Charge every candidate execution (including duplicates), stop at budget.
    """
    levels={1:[]}; seen={}; attempts=0
    def offer(value, expr, cost):
        nonlocal attempts
        attempts+=1
        if value in seen: return None
        seen[value]=expr; levels.setdefault(cost,[]).append((value,expr))
        return (expr,attempts) if value==target else None
    for expr,v in list(library)+list(ATOM.items()):
        r=offer(v,expr,1)
        if r:return r
    for cost in range(2,max_cost+1):
        for v,e in levels.get(cost-1,[]):
            if attempts>=budget:return None,attempts
            r=offer(v^FULL,f'~({e})',cost)
            if r:return r
        for i in range(1,cost//2+1):
            j=cost-1-i
            if i>j:continue
            for a,ea in levels.get(i,[]):
                for b,eb in levels.get(j,[]):
                    if i==j and a>b:continue
                    for op,f in OPS.items():
                        if attempts>=budget:return None,attempts
                        r=offer(f(a,b),f'({ea}{op}{eb})',cost)
                        if r:return r
    return None,attempts

def curriculum(order):
    a,b,c,d,e=[ATOM[f'x{i}'] for i in order]
    # Learner receives only truth table, not expression or ingredients.
    p=(a^b)&c
    q=(p|d)^e
    r=(q&b)|p
    s=(r^a)&e
    null=(a&d)^(b|e)
    return [p,q,r,s,null]

In [ ]:
for order in [(0,1,2,3,4),(4,2,0,1,3)]:
    print('curriculum',order)
    lib=[]; total_flat=total_lib=0
    for i,t in enumerate(curriculum(order)):
        start=perf_counter(); f,fc=synth(t); ft=perf_counter()-start
        start=perf_counter(); m,mc=synth(t,lib); mt=perf_counter()-start
        total_flat+=fc; total_lib+=mc
        print(i,'flat',f,fc,'library',m,mc,'seconds',round(ft,4),round(mt,4))
        if m is not None and i<4: lib.append((f'K{i}',t))
    print('total executed candidates',total_flat,total_lib)

### Captured standalone CPU output — E6
First curriculum (second is similar):
| Task | Flat candidate attempts | Library candidate attempts |
|---|---:|---:|
| K0 | 433, solved | 433, solved |
| K1 | 25,000, failed | 1,019, solved |
| K2 | 25,000, failed | 504, solved |
| K3 | 25,000, failed | 2,315, solved |
| unrelated | 7,956, solved | 25,000, failed |

The actual discovered definitions were K0 = x2 & (x0 ^ x1), K1 = x4 ^ (K0 | x3), K2 = K0 | (K1 & x1), K3 = x4 & (K2 ^ x0). K2 refers to K1 which refers to K0; not just repeated execution of the same layer. The library is still memorizing solved functions, not discovering parameterized common subexpressions across unrelated tasks. This is the simplest executable construction, not a novel program synthesis algorithm.

### Repair negative transfer, then test on fresh permutations
Exploratory intervention chosen **after** seeing the unrelated-task failure: allocate 5k candidate attempts to library search, then 20k to flat search if needed. Same 25k total budget. Two fresh permutations are repetitions of the same generator, not independent domains. Store the evaluated internally generated definition, not the supplied table; verify exact equality. Do not interpret the heuristic scheduler as a learned metarule.

In [ ]:
for order in [(3,0,4,1,2),(2,4,1,0,3)]:
    lib=[]; env=dict(ATOM); counts=[0,0]; solved=[0,0]
    print('fresh curriculum',order)
    for i,t in enumerate(curriculum(order)):
        f,fc=synth(t)
        if lib:
            m,mc=synth(t,lib,budget=5000)
            if m is None:
                m,extra=synth(t,budget=20000);mc+=extra
        else: m,mc=synth(t)
        assert mc<=25000
        for j,(expr,cost) in enumerate([(f,fc),(m,mc)]):
            counts[j]+=cost;solved[j]+=expr is not None
        if m is not None:
            value=eval(m,{'__builtins__':{}},env)&FULL # only internally generated expressions
            assert value==t
            if i<4: env[f'K{i}']=value;lib.append((f'K{i}',value))
        print(i,'flat',f,fc,'hybrid',m,mc)
    print('solved',solved,'candidate counts',counts)

Captured outputs: flat solved 2/5 vs hybrid 5/5, candidate counts 83,992 vs 17,101 on first fresh permutation and 83,236 vs 16,729 on second. This is an engineered curriculum, not evidence of superiority to DreamCoder or Stitch. The fallback is hand-designed and budgets were selected after exploration.

## E7 · Next falsifiable research question (not implemented)
Combine the mechanisms rather than scale either toy:
1. Represent each definition as an immutable typed AST with stable version and dependency references. Each task has immutable observations and candidate programs; infer a compact library from repeated evidence, not only whole solved tasks.
2. A contradiction triggers candidate **theory edits**: revise a shared definition, add a scoped exception, change rule priority, split a definition, or introduce a new abstraction. Each candidate re-executes affected descendants against prior evidence and new observations. Keep competing theories when evidence is insufficient; never retroactively change recorded predictions.
3. Treat successful edits themselves as executable typed AST-to-AST programs. Mine parameterized compositions from independently observed edit histories; use these as proposals for future theory revisions. Learn a policy allocating search budget across library, primitive and edit search. A fixed scheduler is a baseline.
4. Test delayed counterexamples on at least two independently implemented generators; hold out entire compositions, introduce noisy evidence and out-of-library tasks. Compare against semantic-cached flat synthesis, retrieval, library learning/MDL (DreamCoder/Stitch-style), predicate invention, fixed Bayesian theory revision, and their combination. Equal candidate execution and end-to-end wall-clock budgets including training/library construction and validation. Report cumulative error, invalidated old rules, evidence needed for recovery, library description length and search cost. Freeze test sets and budgets before execution; no tuning on held-out results.
5. Only then integrate as a selective correction to WARC with strong neural fallback. Test same causal recordings, full WARC TTT baseline, exact prediction, false edits and change F1, then official evaluator if available. This notebook does not establish an ARC leaderboard claim.

**Kill criterion:** if learned theory-edit operators fail to beat ordinary library learning plus conventional belief revision on independent tasks under the same total budget, there is no demonstrated reason to scale this architecture. Pāṇinian inspiration is a source of hypotheses, not evidence of performance.